<!-- NB_DOC_INTRO_V1 -->
> 📚 **Documentation thematique** : [docs/06_TS_TDS.md](docs/06_TS_TDS.md) (Ts Tds)
> 📖 **Inventaire global** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md)
> 🗂️ **README projet** : [README.md](README.md)

## A quoi sert ce notebook

Util `time_seq_generator` : transforme une serie en `(X, y)` sliding window pour LSTM/GRU/Transformer.

---


# TS — Générer des séquences pour LSTM/GRU/Transformer

> **Objectif** : transformer une série temporelle 1D (ou multivariée) en `(X, y)` consommable par un modèle séquentiel.

## Pourquoi ce notebook ?

Un modèle séquentiel (LSTM, GRU, TCN, Transformer) prend en entrée des **fenêtres temporelles** et prédit une (ou plusieurs) valeur(s) future(s).

```
Série brute :    [v1, v2, v3, v4, v5, v6, v7, v8, v9, v10]
                  ▲────window=3────▲
                                   └─horizon=1─▶ y = v4

Sliding window génère :
  X[0] = [v1, v2, v3]  →  y[0] = v4
  X[1] = [v2, v3, v4]  →  y[1] = v5
  X[2] = [v3, v4, v5]  →  y[2] = v6
  ...
  X[N-window] = [v_{N-window}, ..., v_{N-1}] →  y[N-window] = v_N
```

## Choix critiques

| Paramètre | Effet | Recommandation |
|---|---|---|
| **`window`** (taille de la fenêtre) | Combien de pas passés le modèle voit | 1× à 2× la période saisonnière |
| **`horizon`** | Combien de pas dans le futur on prédit | 1 (single-step) le plus simple |
| **`step`** | Stride entre 2 fenêtres consécutives | 1 (max données) ou égal à `horizon` (pas de chevauchement) |
| **Multi-output** | Prédire les `h` prochains au lieu d'1 | `y.shape = (N, horizon)` |
| **Multivarié** | Plusieurs séries comme features | `X.shape = (N, window, n_features)` |

## Méthodes équivalentes natives

- **Keras** : `keras.utils.timeseries_dataset_from_array(data, targets, sequence_length=window, ...)`
- **PyTorch** : Subclass `torch.utils.data.Dataset` (cf. [DL_PyTorch](./DL_PyTorch))
- **`darts`** : `TimeSeriesSplit` natif sur `TimeSeries` objects.
- **`tsai`** : `SlidingWindow` (sur Fastai).

> ⚠️ **Anti-pattern courant** : générer les fenêtres AVANT le train/test split → **data leakage** (le test contient des features qui chevauchent le train). Toujours splitter d'abord la série, puis générer.


In [ ]:
import pandas as pd
import numpy as np
# Création du DataFrame
df = pd.DataFrame({
    'X1': list(range(30)),
    'X2': list(range(3)) * 10,
    'y': list(range(29, -1, -1))
})

# Affichage du DataFrame
df.head(2)

,X1,X2,y
0,0,0,29
1,1,1,28


In [ ]:
seq_length = 4
overlap = 2
pos = 0

def time_seq_generator(df, seq_length, seq_cols, target , overlap, pos):
    sequences, labels = [], []
    df = df.iloc[::-1].reset_index(drop=True)
    for i in range(len(df) - seq_length):
        if not i % overlap:
            X_values , y_labels= df[seq_cols].values, df[target].values
            sequences.append(X_values[i:i+seq_length])
            labels.append(y_labels[i-pos] )
    X_result = np.array(sequences)
    X_result = X_result[::-1, ::-1, ...]
    y_result = np.array(labels).reshape(-1, 1)
    y_result = y_result[::-1, ::-1, ...]
    return X_result, y_result

X, y  =  time_seq_generator(df, seq_length, ['X1', 'X2'], "y" , 2, pos)

In [ ]:
X[0:2]

array([[[2, 2],
        [3, 0],
        [4, 1],
        [5, 2]],

       [[4, 1],
        [5, 2],
        [6, 0],
        [7, 1]]])

In [ ]:
y[0:2]

array([[24],
       [22]])

In [ ]:
df.head(20)

,X1,X2,y
0,0,0,29
1,1,1,28
2,2,2,27
3,3,0,26
4,4,1,25
5,5,2,24
6,6,0,23
7,7,1,22
8,8,2,21
9,9,0,20


---

## Variantes utiles

### Multi-output (prédire H pas)
```python
def time_seq_multistep(series, window, horizon):
    X, Y = [], []
    for i in range(len(series) - window - horizon + 1):
        X.append(series[i : i + window])
        Y.append(series[i + window : i + window + horizon])
    return np.array(X), np.array(Y)  # X: (N, window), Y: (N, horizon)
```

### Multivarié (n features) + cible séparée
```python
def time_seq_multivariate(features, target, window, horizon=1):
    """features: (T, F), target: (T,). Retourne X (N, window, F), y (N,)."""
    X, y = [], []
    for i in range(len(features) - window - horizon + 1):
        X.append(features[i : i + window])
        y.append(target[i + window + horizon - 1])
    return np.array(X), np.array(y)
```

### Walk-forward validation (pas de leakage)
```python
n = len(series)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_series = series[:train_end]
val_series = series[train_end - window : val_end]   # on garde la window de contexte
test_series = series[val_end - window :]

X_train, y_train = time_seq_generator(train_series, window, horizon)
X_val, y_val = time_seq_generator(val_series, window, horizon)
X_test, y_test = time_seq_generator(test_series, window, horizon)
```

### Avec scaler (sans leakage)
```python
from sklearn.preprocessing import StandardScaler

# Fit sur TRAIN uniquement
scaler = StandardScaler().fit(train_series.reshape(-1, 1))
train_scaled = scaler.transform(train_series.reshape(-1, 1)).ravel()
val_scaled = scaler.transform(val_series.reshape(-1, 1)).ravel()
test_scaled = scaler.transform(test_series.reshape(-1, 1)).ravel()

# Puis sliding window sur chaque
```

## Voir aussi

- [TS_Time_Series_Overview.ipynb](./TS_Time_Series_Overview.ipynb) — feature engineering complet sur TS
- [TS_Maintenance_Predictive_GOOD.ipynb](./TS_Maintenance_Predictive_GOOD.ipynb) — usage en production (engineering features par moteur)
- [DL_PyTorch](./DL_PyTorch) — `Dataset` custom pour TS
